# APAN 5200 — Ensemble Models Mock Quiz 2
**Dataset:** `spotify_data.csv` | **Outcome:** `rating` (continuous)

Same structure as Question 8. **Configuration:** `n_estimators=100`, `random_state=2024`

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    VotingRegressor, BaggingRegressor,
    RandomForestRegressor, AdaBoostRegressor,
    GradientBoostingRegressor
)
from sklearn.metrics import mean_squared_error

In [ ]:
# ============================================================
# Q1: Read spotify_data.csv. Drop: id, performer, song, genre, key.
#     y = rating, X = remaining features.
#     y_binned = pd.qcut(data.rating, q=20)
#     Stratified split: 70% train / 30% test, random_state=2024
#
# QUESTION: What is the median TEMPO in the train sample?
# ANSWER  : 118.352
# ============================================================

data = pd.read_csv('spotify_data.csv')
data = data.drop(columns=['id','performer','song','genre','key'])
y = data['rating']
X = data.drop(columns=['rating'])

y_binned = pd.qcut(data.rating, q=20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=2024, stratify=y_binned
)

print('Train size:', X_train.shape[0], '| Test size:', X_test.shape[0])
print('Median tempo (train):', X_train['tempo'].median())  # ✅ 118.352

In [ ]:
# ============================================================
# Q2: Construct a LINEAR REGRESSION using ONLY genre_ dummies.
# QUESTION: What is the RMSE in the train sample?
# ANSWER  : 15.7915
# ============================================================

genre_cols = [c for c in X.columns if c.startswith('genre_')]
lr = LinearRegression()
lr.fit(X_train[genre_cols], y_train)
rmse_lr = np.sqrt(mean_squared_error(y_train, lr.predict(X_train[genre_cols])))
print(f'Linear Regression (genre dummies) RMSE (train): {rmse_lr:.4f}')  # ✅ 15.7915

In [ ]:
# ============================================================
# Q3: Construct a TREE MODEL using genre_ dummies. Use max_depth=3.
# QUESTION: What is the RMSE in the train sample?
# ANSWER  : 15.8199
# ============================================================

tree = DecisionTreeRegressor(max_depth=3, random_state=2024)
tree.fit(X_train[genre_cols], y_train)
rmse_tree = np.sqrt(mean_squared_error(y_train, tree.predict(X_train[genre_cols])))
print(f'Tree (max_depth=3, genre) RMSE (train): {rmse_tree:.4f}')  # ✅ 15.8199

In [ ]:
# ============================================================
# Q4: Combine the two models above using a VOTING model.
# QUESTION: What is the train sample RMSE?
# ANSWER  : 15.7479
# ============================================================

voting = VotingRegressor(estimators=[
    ('lr',   LinearRegression()),
    ('tree', DecisionTreeRegressor(max_depth=3, random_state=2024))
])
voting.fit(X_train[genre_cols], y_train)
rmse_voting = np.sqrt(mean_squared_error(y_train, voting.predict(X_train[genre_cols])))
print(f'Voting RMSE (train): {rmse_voting:.4f}')  # ✅ 15.7479

In [ ]:
# ============================================================
# Q5: Construct a BAGGING model with 100 bootstrapped samples.
#     Fit tree estimators using ALL predictors. Set max_depth=8.
# QUESTION: What is the RMSE in the train sample?
# ANSWER  : 14.1318
# ============================================================

bagging_tree = BaggingRegressor(
    estimator=DecisionTreeRegressor(max_depth=8),
    n_estimators=100,
    random_state=2024,
    oob_score=True
)
bagging_tree.fit(X_train, y_train)
rmse_bag_tree = np.sqrt(mean_squared_error(y_train, bagging_tree.predict(X_train)))
print(f'Bagging Tree (md=8, n=100) RMSE (train): {rmse_bag_tree:.4f}')  # ✅ 14.1318
print(f'Bagging Tree OOB R²: {bagging_tree.oob_score_:.4f}')            # ✅ 0.1673

In [ ]:
# ============================================================
# Q6: Construct a BAGGING model with 100 bootstrapped samples,
#     fit a LINEAR REGRESSION with ALL predictors.
# QUESTION: What is the RMSE in the train sample?
# ANSWER  : 15.3434
# ============================================================

bagging_lr = BaggingRegressor(
    estimator=LinearRegression(),
    n_estimators=100,
    random_state=2024,
    oob_score=True
)
bagging_lr.fit(X_train, y_train)
rmse_bag_lr = np.sqrt(mean_squared_error(y_train, bagging_lr.predict(X_train)))
print(f'Bagging LR (n=100) RMSE (train): {rmse_bag_lr:.4f}')  # ✅ 15.3434
print(f'Bagging LR OOB R²: {bagging_lr.oob_score_:.4f}')       # ✅ 0.1433

In [ ]:
# ============================================================
# Q7: Fit a RANDOM FOREST model with 100 trees. Use ALL predictors.
# QUESTION: What is the RMSE in the train sample?
# ANSWER  : 5.7007
# ============================================================

rf = RandomForestRegressor(n_estimators=100, random_state=2024, oob_score=True)
rf.fit(X_train, y_train)
rmse_rf = np.sqrt(mean_squared_error(y_train, rf.predict(X_train)))
print(f'Random Forest (n=100) RMSE (train): {rmse_rf:.4f}')  # ✅ 5.7007
print(f'Random Forest OOB R²: {rf.oob_score_:.4f}')           # ✅ 0.1483

In [ ]:
# ============================================================
# Q8: Based on the Random Forest above, which feature is MOST important?
# ANSWER  : track_duration
# ============================================================

feat_imp = pd.Series(rf.feature_importances_, index=X_train.columns)\
             .sort_values(ascending=False)
print('Top 5 Feature Importances (Random Forest):')
print(feat_imp.head(5).to_string())  # ✅ track_duration top

In [ ]:
# ============================================================
# Q9: Which model has the LOWEST OOB score?
# ANSWER  : Bagging LR (OOB ≈ 0.1433)
#
# KEY INSIGHT: Bagging LR has the lowest OOB R² because linear
# regression is a low-variance model — bagging provides little
# additional benefit, and the model is constrained by its
# structural assumptions.
# ============================================================

print('OOB R² Scores:')
print(f'  Bagging Tree (md=8, n=100): {bagging_tree.oob_score_:.4f}')
print(f'  Bagging LR   (n=100)      : {bagging_lr.oob_score_:.4f}  ← LOWEST')
print(f'  Random Forest (n=100)     : {rf.oob_score_:.4f}')

In [ ]:
# ============================================================
# Q10: Construct a BOOSTING model using ADABOOST with a tree
#      estimator. Use 100 estimators. Use ALL predictors.
# QUESTION: What is the RMSE in the train sample?
# ANSWER  : 0.6662
# ============================================================

ada = AdaBoostRegressor(
    estimator=DecisionTreeRegressor(),
    n_estimators=100,
    random_state=2024
)
ada.fit(X_train, y_train)
rmse_ada = np.sqrt(mean_squared_error(y_train, ada.predict(X_train)))
print(f'AdaBoost (n=100) RMSE (train): {rmse_ada:.4f}')  # ✅ 0.6662

In [ ]:
# ============================================================
# Q11: Construct a GRADIENT BOOSTING model with 100 trees.
#      Use ALL predictors.
# QUESTION: What is the RMSE in the train sample?
# ANSWER  : 14.7614
# ============================================================

gb = GradientBoostingRegressor(n_estimators=100, random_state=2024)
gb.fit(X_train, y_train)
rmse_gb = np.sqrt(mean_squared_error(y_train, gb.predict(X_train)))
print(f'Gradient Boosting (n=100) RMSE (train): {rmse_gb:.4f}')  # ✅ 14.7614

In [ ]:
# ============================================================
# Q12: Construct a STOCHASTIC GRADIENT BOOSTING model with 100 trees.
#      Set max_features=0.3 and subsample=0.5. Use ALL predictors.
# QUESTION: What is the RMSE in the train sample?
# ANSWER  : 14.8337
# ============================================================

sgb = GradientBoostingRegressor(
    n_estimators=100,
    max_features=0.3,
    subsample=0.5,
    random_state=2024
)
sgb.fit(X_train, y_train)
rmse_sgb = np.sqrt(mean_squared_error(y_train, sgb.predict(X_train)))
print(f'Stochastic GB (n=100, mf=0.3, sub=0.5) RMSE (train): {rmse_sgb:.4f}')  # ✅ 14.8337

In [ ]:
# ============================================================
# Q13: Based on the Stochastic GB above, which is the most important feature?
# ANSWER: track_duration
# ============================================================

feat_imp_sgb = pd.Series(sgb.feature_importances_, index=X_train.columns)\
                 .sort_values(ascending=False)
print('Top 5 Feature Importances (Stochastic GB):')
print(feat_imp_sgb.head(5).to_string())  # ✅ track_duration top

---
## Answer Summary

In [ ]:
summary = {
    'Q1  Median tempo (train)':                  '118.352 BPM',
    'Q2  LR (genre dummies) RMSE train':         '15.7915',
    'Q3  Tree (md=3, genre) RMSE train':         '15.8199',
    'Q4  Voting RMSE train':                      '15.7479',
    'Q5  Bagging Tree (md=8, n=100) RMSE':       '14.1318',
    'Q6  Bagging LR (n=100) RMSE':               '15.3434',
    'Q7  Random Forest (n=100) RMSE':            '5.7007',
    'Q8  RF most important feature':              'track_duration',
    'Q9  Lowest OOB score model':                 'Bagging LR (0.1433)',
    'Q10 AdaBoost (n=100) RMSE':                 '0.6662',
    'Q11 Gradient Boosting (n=100) RMSE':        '14.7614',
    'Q12 Stochastic GB RMSE':                    '14.8337',
    'Q13 SGB most important feature':             'track_duration',
}
for k,v in summary.items():
    print(f'{k:<48} → {v}')